---

Database storage

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2
from pgl import pglDatabase, pglSession, pglDataset

---

Display all sessions in database

---

In [ ]:
pglDatabase.print()

---

Load a behavioral session

---

In [ ]:
session = pglSession.loadSession(104)

In [ ]:
session.history()

---

print session

---

In [ ]:
session.print()

In [ ]:
session.mne.displayReport()


---

Get behavioral data

---

In [ ]:
trials = session.getBehaviorTable("memoryDescriptionTask")

In [ ]:
# print a row of the table
#print(trials)
print(trials.iloc[1])

In [ ]:
# compute response times relative to a specific segment
trials = session.getBehaviorTable(taskName="memoryDescriptionTask", responseReference="segment", segmentNum=1)

---

Create a dataset

---

In [ ]:
dataset = pglDataset(name="Digital Brain Memory Pilot")
dataset.addSession([102,103,106,107,100,101,96,97,98,99,104,105])

---

get the behavioral table

---

In [ ]:
trials = dataset.getBehaviorTable(taskName="memoryDescriptionTask")

In [ ]:
# print a trial
print(trials.iloc[1000])

---

Save the dataset

---

In [ ]:
dataset.save()

---

Print datasets

---

In [ ]:
pglDatabase.print(datasets=True)

---

Load a dataset

---

In [ ]:
dataset = pglDataset.load(6)
dataset.print()

In [3]:
import ast
import difflib
import os
from pathlib import Path


DEFAULT_SKIP_DIRS = {
    ".git", ".hg", ".svn", ".venv", "venv", "env",
    "__pycache__", ".tox", ".mypy_cache", ".pytest_cache",
    "build", "dist", "node_modules", ".ipynb_checkpoints",
}


def iter_python_files(root, skip_dirs=None):
    """Walk Python source files without following directory symlinks."""
    root = Path(root).resolve()
    excluded = DEFAULT_SKIP_DIRS | set(skip_dirs or ())

    for directory, dirs, files in os.walk(root, followlinks=False):
        dirs[:] = sorted(
            name for name in dirs
            if name not in excluded
            and not Path(directory, name).is_symlink()
        )
        for name in sorted(files):
            path = Path(directory, name)
            if path.suffix == ".py" and not path.is_symlink():
                yield path


def rewrite_pgl_references(source, moves):
    """
    Return (updated_source, replacement_count).

    moves maps method names to their new owner class/module names.

    Supports:
        from .pglBase import pglBase
        from pgl.pglBase import pglBase
        from pgl import pglBase

    Aliased imports and other access patterns are left for review.
    """
    tree = ast.parse(source)

    imports = [
        node for node in ast.walk(tree)
        if isinstance(node, ast.ImportFrom)
        and any(
            item.name == "pglBase" and item.asname is None
            for item in node.names
        )
    ]

    references = [
        node for node in ast.walk(tree)
        if isinstance(node, ast.Attribute)
        and isinstance(node.value, ast.Name)
        and node.value.id == "pglBase"
        and node.attr in moves
        and isinstance(node.ctx, ast.Load)
    ]

    if not references:
        return source, 0

    if not imports:
        raise ValueError(
            "Found pglBase references but no supported pglBase import; "
            "review this file manually."
        )

    # AST column offsets are UTF-8 byte offsets.
    data = source.encode("utf-8")
    lines = data.splitlines(keepends=True)
    starts = []
    position = 0
    for line in lines:
        starts.append(position)
        position += len(line)

    def span(node):
        start = starts[node.lineno - 1] + node.col_offset
        end = starts[node.end_lineno - 1] + node.end_col_offset
        return start, end

    owners = sorted({moves[node.attr] for node in references})
    replaced_receivers = {id(node.value) for node in references}

    # Keep the old imports if pglBase is used for anything else,
    # e.g. a superclass or a graphics method.
    keep_base = any(
        isinstance(node, ast.Name)
        and node.id == "pglBase"
        and id(node) not in replaced_receivers
        for node in ast.walk(tree)
    )

    edits = []

    for node in references:
        start, end = span(node)
        replacement = f"{moves[node.attr]}.{node.attr}".encode("utf-8")
        edits.append((start, end, replacement))

    newline = b"\r\n" if b"\r\n" in data else b"\n"

    for node in imports:
        module = node.module or ""

        if module == "pglBase" or module.endswith(".pglBase"):
            prefix = "." * node.level + module[:-len("pglBase")]
        elif module == "pgl":
            prefix = "." * node.level + module + "."
        else:
            raise ValueError(
                f"Unsupported import on line {node.lineno}: "
                f"{ast.get_source_segment(source, node)}"
            )

        start, end = span(node)
        indent = lines[node.lineno - 1][:node.col_offset]
        tail = lines[node.end_lineno - 1][node.end_col_offset:].strip()

        # Avoid changing imports embedded in one-line suites or
        # semicolon-separated statements.
        if indent.strip() or (tail and not tail.startswith(b"#")):
            raise ValueError(
                f"Import on line {node.lineno} is not a standalone "
                "statement; review it manually."
            )

        new_imports = [
            f"from {prefix}{owner} import {owner}".encode("utf-8")
            for owner in owners
        ]

        # For mixed imports, retain the original conservatively.
        retain_original = keep_base or len(node.names) != 1
        pieces = ([data[start:end]] if retain_original else []) + new_imports
        replacement = (newline + indent).join(pieces)

        edits.append((start, end, replacement))

    for start, end, replacement in sorted(edits, reverse=True):
        data = data[:start] + replacement + data[end:]

    updated = data.decode("utf-8")
    ast.parse(updated)  # Do not return a syntactically broken edit.
    return updated, len(references)


def plan_pgl_refactor(root, moves, skip_dirs=None):
    """Build a dry-run plan; do not write any files."""
    root = Path(root).resolve()
    changes = []
    issues = []

    for path in iter_python_files(root, skip_dirs):
        original = path.read_bytes()

        try:
            # Preserve a UTF-8 BOM if present.
            encoding = "utf-8-sig" if original.startswith(b"\xef\xbb\xbf") else "utf-8"
            source = original.decode(encoding)
            updated, count = rewrite_pgl_references(source, moves)

            if updated != source:
                changes.append({
                    "path": path,
                    "before": original,
                    "after": updated.encode(encoding),
                    "count": count,
                    "encoding": encoding,
                })
        except (UnicodeError, SyntaxError, ValueError) as error:
            issues.append((path, str(error)))

    print(
        f"Planned {sum(item['count'] for item in changes)} replacements "
        f"in {len(changes)} files."
    )

    for path, message in issues:
        print(f"REVIEW {path.relative_to(root)}: {message}")

    return changes, issues


def show_pgl_refactor(changes):
    """Display unified diffs for the proposed edits."""
    for item in changes:
        encoding = item["encoding"]
        before = item["before"].decode(encoding).splitlines(keepends=True)
        after = item["after"].decode(encoding).splitlines(keepends=True)

        print("".join(difflib.unified_diff(
            before,
            after,
            fromfile=str(item["path"]),
            tofile=str(item["path"]) + " (proposed)",
        )), end="")


def apply_pgl_refactor(changes, backup_suffix=".before-pgl-refactor"):
    """Apply a reviewed plan, refusing stale plans or existing backups."""
    # Preflight the whole plan before writing anything.
    for item in changes:
        path = item["path"]
        backup = path.with_name(path.name + backup_suffix)

        if path.read_bytes() != item["before"]:
            raise RuntimeError(f"File changed since planning: {path}")

        if backup.exists():
            raise FileExistsError(f"Backup already exists: {backup}")

    for item in changes:
        path = item["path"]
        backup = path.with_name(path.name + backup_suffix)
        backup.write_bytes(item["before"])
        path.write_bytes(item["after"])

    print(f"Updated {len(changes)} files; backups saved beside originals.")


def find_remaining_pgl_references(root, method_names, skip_dirs=None):
    """
    Report remaining attribute references to moved methods.

    This also catches candidates such as self.validateFilesystem,
    cls.validateFilesystem, or an aliased pglBase reference.
    Not every reported reference necessarily needs changing.
    """
    method_names = set(method_names)

    for path in iter_python_files(root, skip_dirs):
        try:
            source = path.read_bytes().decode("utf-8-sig")
            tree = ast.parse(source)
        except (UnicodeError, SyntaxError):
            continue

        for node in ast.walk(tree):
            if not isinstance(node, ast.Attribute):
                continue
            if node.attr not in method_names:
                continue

            # Exclude references already using the intended new owners.
            if (
                isinstance(node.value, ast.Name)
                and node.value.id in {"pglStorage", "pglInfo"}
            ):
                continue

            expression = ast.get_source_segment(source, node)
            print(f"{path}:{node.lineno}: {expression}")

In [5]:
import subprocess
import sys
import textwrap

from pathlib import Path
root = Path("/Users/justin/proj/pgl")

test_code = textwrap.dedent("""
    from pathlib import Path
    from tempfile import TemporaryDirectory
    from fsspec.implementations.local import LocalFileSystem
    from pgl.pglStorage import pglStorage

    assert pglStorage.makeValidFilename(" Hello / World! ") == "hello_world"
    assert pglStorage.makeValidFilename("...") == "unnamed"

    with TemporaryDirectory() as directory:
        filesystem, path, prefix = pglStorage.validateFilesystem(dataPath=directory)
        assert isinstance(filesystem, LocalFileSystem)
        assert filesystem.exists(path)

        supplied = LocalFileSystem()
        filesystem, path, prefix = pglStorage.validateFilesystem(filesystem=supplied, dataPath=directory)
        assert filesystem is supplied

        missing = str(Path(directory) / "missing")
        filesystem, path, prefix = pglStorage.validateFilesystem(dataPath=missing, requireExists=False)
        assert isinstance(filesystem, LocalFileSystem)
        assert not filesystem.exists(path)

        filesystem, path, prefix = pglStorage.validateFilesystem(dataPath=missing, create=True)
        assert filesystem.isdir(path)

    print("pglStorage import and local smoke tests passed.")
""")

result = subprocess.run([sys.executable, "-c", test_code], cwd=root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

pglStorage import and local smoke tests passed.


Exit code: 0


In [7]:
find_remaining_pgl_references(root, {"validateFilesystem", "makeValidFilename"})